In [1]:
import cflinstance
import utils
import numpy as np

In [11]:
instances_and_sols = [cflinstance.CFLInstance.load_instance_and_solution(f"{utils.Constants.instancesDatasetPath}/instance_{i}.npz") for i in range(50)]
instances = [inst["instance"] for inst in instances_and_sols]
solutions = [sol["solution"] for sol in instances_and_sols]

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
def generate_dataset(instances, solutions):
    dataset_x = []
    dataset_y = []
    
    for instance, solution in zip(instances, solutions):
        # for each facility, build an array with each of its associated features
        features = instance.compute_features()
              
        reshaped_features = np.array(list(features.values())).T
        y_solution = abs(solution["y"])
        dataset_x.append(reshaped_features)
        dataset_y.append(y_solution)
        
    dataset_x = torch.tensor(np.concatenate(dataset_x, axis=0), dtype=torch.float32)
    dataset_y = torch.tensor(np.concatenate(dataset_y, axis=0), dtype=torch.float32)
    return dataset_x, dataset_y

In [14]:
X, y = generate_dataset(instances, solutions)
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

In [31]:
model = nn.Linear(X.shape[1], 1)

def compute_loss(thetas, instances: list[cflinstance.CFLInstance], y_true, n_rep = 15):
    fy_score = - torch.dot(thetas.reshape(-1), y_true)
    true_loss = torch.tensor(0.0, dtype=torch.float32)
    idx = 0
    for instance in instances:
        # idx jsqu'a idx + instance.n_facilities
        inst_thetas = thetas[idx: idx + instance.n_facilities].detach().numpy().reshape(-1)
        
        # compute E[O.y] where O is noised thetas and y is the solution of the model with those noised thetas
        esp = torch.tensor(0.0, dtype=torch.float32)
        
        for _ in range(n_rep):
            noised_thetas = inst_thetas + np.random.normal(0, 0.2, size=inst_thetas.shape)
            thetaed_model = instance.get_solved_model_using_thetas(-noised_thetas, timeout=25e-3)

            _, y_vals = utils.parse_vars(thetaed_model.getVars(), instance.n_facilities, instance.n_clients)
            y_vals = torch.tensor([v.X for v in y_vals], dtype=torch.float32)
            esp = esp + torch.dot(y_vals, torch.tensor(inst_thetas, dtype=torch.float32))
            true_loss = true_loss + ((y_vals - y_true[idx: idx + instance.n_facilities])**2).sum()
                    
        # esp = np.max(esp, axis=0)
        fy_score = fy_score + esp / n_rep
    
        idx += instance.n_facilities
    return fy_score, true_loss

optimizer = optim.SGD(model.parameters(), lr=1e-3)

# training loop
losses = []
for epoch in range(10):
    # for batch_X, batch_y in dataloader:
        pred = model(X)
        # print("pred", pred)
        loss, true_loss = compute_loss(pred, instances, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, True Loss: {true_loss.item():.4f}")
        losses.append(loss.item())

print("final loss:", loss.item())

Epoch 1, Loss: 105.7878, True Loss: 2362.0000
Epoch 2, Loss: 3314.5688, True Loss: 2286.0000
Epoch 3, Loss: 3592.3464, True Loss: 1679.0000
Epoch 4, Loss: 7171.3394, True Loss: 1941.0000
Epoch 5, Loss: 9103.6152, True Loss: 1955.0000
Epoch 6, Loss: 7718.2588, True Loss: 1674.0000
Epoch 7, Loss: 10548.3174, True Loss: 1731.0000
Epoch 8, Loss: 23617.4395, True Loss: 2256.0000
Epoch 9, Loss: 19962.5977, True Loss: 1958.0000
Epoch 10, Loss: 15708.3174, True Loss: 1707.0000
final loss: 15708.3173828125


In [30]:
vals = []
for inst, sol in zip(instances, solutions):
    features = inst.compute_features()
    reshaped_features = np.array(list(features.values())).T
    X_inst = torch.tensor(reshaped_features, dtype=torch.float32)
    pred = model(X_inst).detach().numpy().reshape(-1)
    zipped = list(zip(pred, sol["y"], inst.data["opening_costs"]))
    vals.extend(zipped)

for val in vals:
    print(f"{val[0]}\t {abs(val[1])}\t {val[2]}")


228.56842041015625	 1.0	 3.152179506283149
177.52606201171875	 1.0	 1.1944885749156688
387.6206970214844	 1.0	 4.144091619117079
150.98733520507812	 0.0	 3.612830328836671
457.84490966796875	 0.0	 9.23929028795887
386.1734619140625	 0.0	 11.565052511914844
212.46620178222656	 0.0	 0.5425149448516339
264.2690124511719	 0.0	 5.784205706511473
205.44052124023438	 1.0	 4.0135928001953936
140.3478240966797	 0.0	 0.16224258251693383
285.2843017578125	 0.0	 9.203590293435202
442.6839599609375	 1.0	 6.76635343939637
264.1623229980469	 1.0	 1.6432731094170163
290.4714660644531	 1.0	 1.5111055858058837
237.15652465820312	 0.0	 1.0058188301952307
267.6673889160156	 0.0	 4.438673738469009
351.3934631347656	 0.0	 10.649214398397742
391.1379699707031	 1.0	 7.324159480095039
509.64215087890625	 0.0	 14.7343816276609
373.6731262207031	 1.0	 2.024420464784823
376.4464416503906	 1.0	 7.810555168510926
198.0094757080078	 0.0	 0.7015832022448478
315.1976623535156	 0.0	 6.400807859236404
271.7600402832031	